# **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# **Load Dataset**

In [ ]:
df = pd.read_csv("CIPLAA.csv")

print(df.shape)
print(df['label'].value_counts())

# **Text Pre Processing**

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d', '', text)
    return text

df['text'] = df['subject'] + " " + df['body']
df['clean_text'] = df['text'].apply(clean_text)

# **Feature Engineering**

In [ ]:
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
text_features = tfidf.fit_transform(df['clean_text'])

# **Numerical Features**

In [ ]:
num_features = df[[
    'num_links', 'num_attachments',
    'has_urgent_words', 'has_spam_words', 'has_phishing_words',
    'contains_html', 'email_length', 'num_exclamations',
    'num_uppercase_words', 'day_of_week', 'hour',
    'is_reply', 'is_forward', 'sender_reputation'
]]

# **Combine Features**

In [ ]:
from scipy.sparse import hstack

X = hstack([text_features, num_features])

# **Encode Labels**

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['label'])

print(dict(zip(le.classes_, range(len(le.classes_)))))

# **Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# **Train Models**

Model 1: Logistic Regression (Strong baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

Model 2: Random Forest (Handles mixed features well)

In [ ]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

# **Evaluation**

Logistic Regression

In [ ]:
y_pred_lr = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

# **Random Forest**

In [ ]:
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

# **Test Custom Email**

In [ ]:
def predict_email(subject, body):
    text = clean_text(subject + " " + body)
    text_vec = tfidf.transform([text])
    dummy = np.zeros((1, num_features.shape[1]))
    from scipy.sparse import hstack
    final_input = hstack([text_vec, dummy])
    
    pred = lr.predict(final_input)
    return le.inverse_transform(pred)[0]


print(predict_email(
    "You won a free iPhone!",
    "Click here to claim your reward now!!!"
))

# **Project Conclusion**

In this project, we built a machine learning model to classify emails by preprocessing text data and extracting features using TF-IDF along with additional email attributes. Models like Logistic Regression and Random Forest were trained and evaluated using standard metrics. The results showed strong performance, especially in detecting spam emails. This demonstrates that combining text processing with machine learning can effectively solve email classification problems.

# **Final Internship Project Submitted By**

In [ ]:
Name:Rabia Shaikh

Domain: Machine Learning

Company Name: Arch Tech

Submission Date: 27 April 2026
